# DELLM Test Notebook

This notebook validates the `DELLMGenerator` implementation in `Classes/dellm_classes.py`.

It includes:
- a mocked smoke test (no API call)
- an optional real inference test (requires `HF_TOKEN`)
- final prompt construction test (`question + schema + DELLM knowledge`)

In [1]:
import json
import os

from Classes.dellm_classes import DELLMGenerator


def pretty(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False))

/Users/nikolajabramov/PycharmProjects/llm4lineage/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:27: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
# Mocked smoke test (no network/API required)
class _DummyChatAdapter:
    def __init__(self, payload):
        self.payload = payload

    def invoke_messages(self, _messages):
        return json.dumps(self.payload)


generator_mock = DELLMGenerator.__new__(DELLMGenerator)
generator_mock.max_retries = 1
generator_mock.max_words = 140
generator_mock.system_prompt = "system"
generator_mock.chat_model = None
generator_mock.chat_adapter = _DummyChatAdapter(
    {
        "knowledge": (
            "Total deposit equals principal deposit amount plus accrued interest. "
            "Method code 12 means PayPal. joined_at is stored as unix timestamp."
        ),
        "categories": [
            "arithmetic_reasoning",
            "domain_terminology",
            "formatting_synonyms",
        ],
    }
)

mock_result = generator_mock.generate_knowledge(
    question="What is total deposits by method for active users?",
    schema={"tables": [{"name": "payments", "columns": [{"name": "joined_at"}]}]},
)

pretty(mock_result)

{
  "knowledge": "Total deposit equals principal deposit amount plus accrued interest. Method code 12 means PayPal. joined_at is stored as unix timestamp.",
  "categories": [
    "arithmetic_reasoning",
    "domain_terminology",
    "formatting_synonyms"
  ]
}


In [3]:
# Sample question + schema for real test
question = "Find monthly total deposits by payment method for active users."

schema = {
    "tables": [
        {
            "name": "users",
            "alias": "u",
            "columns": [
                {"name": "user_id"},
                {"name": "active"},
                {"name": "joined_at"},
            ],
        },
        {
            "name": "payments",
            "alias": "p",
            "columns": [
                {"name": "user_id"},
                {"name": "deposit_amount"},
                {"name": "interest_earned"},
                {"name": "payment_method_code"},
                {"name": "created_at"},
            ],
        },
    ]
}

pretty({"question": question, "schema": schema})

{
  "question": "Find monthly total deposits by payment method for active users.",
  "schema": {
    "tables": [
      {
        "name": "users",
        "alias": "u",
        "columns": [
          {
            "name": "user_id"
          },
          {
            "name": "active"
          },
          {
            "name": "joined_at"
          }
        ]
      },
      {
        "name": "payments",
        "alias": "p",
        "columns": [
          {
            "name": "user_id"
          },
          {
            "name": "deposit_amount"
          },
          {
            "name": "interest_earned"
          },
          {
            "name": "payment_method_code"
          },
          {
            "name": "created_at"
          }
        ]
      }
    ]
  }
}


In [4]:
# Real DELLM call (requires HF_TOKEN in environment)
hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    print("HF_TOKEN is not set. Export HF_TOKEN and rerun this cell for live DELLM inference.")
else:
    generator = DELLMGenerator(
        hf_token=hf_token,
        model=os.environ.get("MODEL_NAME", "Qwen/Qwen3-Coder-30B-A3B-Instruct"),
        provider=os.environ.get("PROVIDER", "scaleway"),
        max_new_tokens=512,
        temperature=0.0,
        max_retries=2,
        max_words=140,
    )

    live_result = generator.generate_knowledge(question=question, schema=schema)
    pretty(live_result)

{
  "knowledge": "Active users are identified by u.active = 1. Monthly totals require extracting year-month from p.created_at using DATE_TRUNC or similar function. Deposits refer to p.deposit_amount which should be summed per month and payment method. Payment method codes need to be mapped to descriptive names if available in a lookup table, otherwise use the raw codes. The query must filter for active users only and group results by month and payment method.",
  "categories": [
    "arithmetic_reasoning",
    "domain_terminology",
    "formatting_synonyms"
  ]
}


In [5]:
# Build final prompt for downstream SQL model
# Uses provided knowledge if available, otherwise auto-generates via DELLM.
manual_knowledge = (
    "Total deposit = deposit_amount + interest_earned. "
    "payment_method_code 12 means PayPal. "
    "joined_at is unix timestamp."
)

prompt_result = generator_mock.build_augmented_prompt(
    question=question,
    schema=schema,
    knowledge=manual_knowledge,
)

print(prompt_result["final_prompt"])
print("\n--- Categories ---")
print(prompt_result["categories"])
print("\n--- Error ---")
print(prompt_result["error"])

User Question:
Find monthly total deposits by payment method for active users.

Database Schema JSON:
{
  "tables": [
    {
      "name": "users",
      "alias": "u",
      "columns": [
        {
          "name": "user_id"
        },
        {
          "name": "active"
        },
        {
          "name": "joined_at"
        }
      ]
    },
    {
      "name": "payments",
      "alias": "p",
      "columns": [
        {
          "name": "user_id"
        },
        {
          "name": "deposit_amount"
        },
        {
          "name": "interest_earned"
        },
        {
          "name": "payment_method_code"
        },
        {
          "name": "created_at"
        }
      ]
    }
  ]
}

DELLM Expert Knowledge:
Total deposit = deposit_amount + interest_earned. payment_method_code 12 means PayPal. joined_at is unix timestamp.

--- Categories ---
[]

--- Error ---
None
